# 08 — Build a reliable classifier for your own image folders

**Prerequisites:** notebooks 05–07; know training, validation, test, and class indices.
**Goals:** validate an ImageFolder dataset, check class mappings and duplicate leakage, train
a model, and save the exact label/preprocessing contract required for prediction.
**Time:** 25–40 minutes. The default generates a tiny demonstration dataset from bundled digits,
with no network download. You can then point the same pipeline at your own prepared folders.

This notebook **does not move, split, delete, repair, or overwrite your own images**. It creates
demo files only inside `data/demo_imagefolder`. User images are opened read-only. For a first real
project use two or three clearly defined classes, then inspect examples and mistakes carefully.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter from the ObjectDetectionPractical folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course_utils import seed_everything, get_device, SMOKE_TEST, ARTIFACTS
seed_everything(42)
device = get_device()
print("Device:", device, "| smoke test:", SMOKE_TEST)


In [ ]:
import hashlib
import json
import warnings
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms as T
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from course_utils import DATA_DIR, TinyCNN, fit, evaluate

USE_MY_DATA = False
MY_DATA_ROOT = ROOT / "my_image_dataset"  # Change this only when USE_MY_DATA = True.
DEMO_ROOT = DATA_DIR / "demo_imagefolder"
IMAGE_SIZE = 32


## 1. Prepare separate split folders

`ImageFolder` assigns alphabetical class names to integer indices. All three partitions must
contain exactly the same class folders and at least one image per class.

```text
my_image_dataset/
  train/cat/image001.jpg     train/dog/image101.jpg
  val/cat/image201.jpg       val/dog/image301.jpg
  test/cat/image401.jpg      test/dog/image501.jpg
```

Split by the **independent source** before copying files: photos of the same person, object,
video, or burst should stay in one partition. Randomly splitting nearly identical frames can
greatly inflate test accuracy. The demo starts from the fixed disjoint digits partitions.


In [ ]:
if not USE_MY_DATA:
    # Guard the sole write destination, and never adopt a pre-existing unrelated folder.
    demo_resolved = DEMO_ROOT.resolve()
    assert demo_resolved.parent == DATA_DIR.resolve() and demo_resolved.name == "demo_imagefolder"
    marker = DEMO_ROOT / "generated_by_course.json"
    if DEMO_ROOT.exists() and not marker.exists():
        if any(DEMO_ROOT.iterdir()):
            raise RuntimeError(f"Refusing to use an unmarked nonempty demo folder: {DEMO_ROOT}")
    DEMO_ROOT.mkdir(parents=True, exist_ok=True)
    if not marker.exists():
        marker.write_text(json.dumps({"source": "sklearn digits", "seed": 42,
                                      "purpose": "generated course demo", "version": 1}), encoding="utf-8")
    # Always export the full deterministic demo, even during smoke tests. Original row IDs
    # keep filenames stable when the notebook is rerun with a different compute budget.
    digits = load_digits()
    all_ids = np.arange(len(digits.target))
    train_ids, held_ids = train_test_split(all_ids, test_size=0.30,
                                           stratify=digits.target, random_state=42)
    val_ids, test_ids = train_test_split(held_ids, test_size=0.50,
                                         stratify=digits.target[held_ids], random_state=42)
    for split_name, ids in {"train": train_ids, "val": val_ids, "test": test_ids}.items():
        for index in ids:
            pixels, label = digits.images[index] / 16.0, digits.target[index]
            destination = DEMO_ROOT / split_name / str(int(label)) / f"digit_{index:04d}.png"
            if not destination.resolve().is_relative_to(demo_resolved):
                raise RuntimeError(f"Demo destination escapes its directory: {destination}")
            destination.parent.mkdir(parents=True, exist_ok=True)
            # Exclusive creation makes reruns safe: existing files are never overwritten.
            if not destination.exists():
                picture = Image.fromarray(np.rint(pixels * 255).astype(np.uint8))
                with destination.open("xb") as stream:
                    picture.save(stream, format="PNG")
    dataset_root = DEMO_ROOT
else:
    dataset_root = MY_DATA_ROOT.expanduser().resolve()
    if not dataset_root.is_dir():
        raise FileNotFoundError(f"Prepare your train/val/test folders first: {dataset_root}")
print("Reading images from:", dataset_root)


## 2. Validate before training

We verify images and decode them in a second open, preserving Pillow's decompression-bomb checks.
Failures are reported instead of silently dropping files, because silent drops can alter class
balance or remove a class entirely. This scan can take time on a large dataset.

The demo uses PNG; your images can use the formats accepted by ImageFolder. Fix or remove corrupt
images yourself after reviewing the report, then rerun. This notebook makes no such changes.


In [ ]:
from torchvision.datasets.folder import IMG_EXTENSIONS

def inspect_image(path):
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("error", Image.DecompressionBombWarning)
            with Image.open(path) as image:
                image.verify()
            with Image.open(path) as image:
                image.load()
                if image.width < 2 or image.height < 2:
                    raise ValueError("Image dimensions are too small")
        return None
    except (OSError, ValueError, SyntaxError, Image.DecompressionBombWarning,
            Image.DecompressionBombError) as error:
        return f"{type(error).__name__}: {error}"

bad_files, image_files = [], {}
for split_name in ["train", "val", "test"]:
    split_path = dataset_root / split_name
    if not split_path.is_dir():
        raise FileNotFoundError(f"Missing split directory: {split_path}")
    image_files[split_name] = sorted(p for p in split_path.rglob("*")
                                    if p.is_file() and p.suffix.lower() in IMG_EXTENSIONS)
    if not image_files[split_name]:
        raise ValueError(f"No supported image files in {split_path}")
    for path in image_files[split_name]:
        problem = inspect_image(path)
        if problem is not None:
            bad_files.append({"path": str(path), "problem": problem})
if bad_files:
    display(pd.DataFrame(bad_files))
    raise ValueError("Image validation failed. Review the files; nothing was changed.")
print("Validated file counts:", {s: len(paths) for s, paths in image_files.items()})


## 3. Verify class maps and exact duplicate leakage

A label index must mean the same class everywhere. Hashing detects byte-identical files across
partitions; it cannot detect resized, re-encoded, cropped, or visually near-identical copies.
Dataset collection records and group-aware splitting are still essential.

This lesson resizes every image to 32×32 RGB to keep training light. That can distort aspect ratio
and lose detail; choose a deliberate crop or letterbox policy for a real project.


In [ ]:
transform = T.Compose([T.Resize((IMAGE_SIZE, IMAGE_SIZE)), T.ToTensor()])
image_sets = {split: datasets.ImageFolder(dataset_root / split, transform=transform)
              for split in ["train", "val", "test"]}
class_to_idx = image_sets["train"].class_to_idx
assert len(class_to_idx) >= 2, "Classification needs at least two classes in this lesson"
for split, dataset in image_sets.items():
    assert dataset.class_to_idx == class_to_idx, f"Class mapping mismatch in {split}"
class_names = image_sets["train"].classes
class_counts = pd.DataFrame({s: Counter(ds.targets) for s, ds in image_sets.items()}).fillna(0).astype(int)
class_counts.index = [class_names[i] for i in class_counts.index]
print("Class map:", class_to_idx)
display(class_counts.sort_index())

def file_digest(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

seen_hashes, duplicate_pairs = {}, []
for split, paths in image_files.items():
    for path in paths:
        digest = file_digest(path)
        if digest in seen_hashes and seen_hashes[digest][0] != split:
            duplicate_pairs.append((str(seen_hashes[digest][1]), str(path)))
        seen_hashes.setdefault(digest, (split, path))
if duplicate_pairs:
    display(pd.DataFrame(duplicate_pairs, columns=["first_path", "duplicate_path"]))
    raise ValueError("Exact cross-split duplicates found. Review your split before training.")
print("No byte-identical cross-split duplicates found.")


In [ ]:
generator = torch.Generator().manual_seed(42)
training_sets = image_sets.copy()
if SMOKE_TEST:
    # Limit compute, while retaining examples from every class where available.
    for split, dataset in image_sets.items():
        per_class = 8 if split == "train" else 3
        labels = np.asarray(dataset.targets)
        ids = np.concatenate([np.flatnonzero(labels == label)[:per_class]
                              for label in range(len(class_names))]).tolist()
        training_sets[split] = Subset(dataset, ids)
loaders = {split: DataLoader(dataset, batch_size=32, shuffle=(split == "train"),
                              num_workers=0, generator=generator if split == "train" else None)
           for split, dataset in training_sets.items()}
xb, yb = next(iter(loaders["train"]))
fig, axes = plt.subplots(1, min(8, len(xb)), figsize=(12, 2), squeeze=False)
for ax, image, label in zip(axes.flat, xb[:8], yb[:8]):
    ax.imshow(image.permute(1, 2, 0)); ax.set_title(class_names[label.item()]); ax.axis("off")
plt.show()
print("Batch:", tuple(xb.shape))


## 4. Establish a simple baseline first

The demo digits can be learned with a tiny CNN. For real natural images, this scratch model may
be weak; reuse notebook 07's pretrained backbone and its **matching preprocessing** as your next
experiment. Keep your current splits fixed when comparing models.

Check whether classes are imbalanced. Accuracy can hide failure on a rare class; notebook 09
develops per-class metrics and error analysis.


In [ ]:
seed_everything(42)
model = TinyCNN(num_classes=len(class_names), in_channels=3).to(device)
history = fit(model, loaders["train"], loaders["val"], epochs=2 if SMOKE_TEST else 10,
              lr=0.003, weight_decay=1e-4, device=device)
history_df = pd.DataFrame(history).set_index("epoch")
history_df[["train_loss", "val_loss"]].plot(figsize=(7, 3), title="Your dataset: training and validation")
plt.show()
test_result = evaluate(model, loaders["test"], device=device)
print(f"Selected model test accuracy: {test_result['accuracy']:.1%}")


## 5. Save the prediction contract and run one-file inference

Checkpoint metadata must preserve class order, channel conversion, resize dimensions, and
normalization. An accurate model can produce meaningless predictions if its labels are decoded
with a different alphabetical order. The prediction below reuses the exact evaluation transform.
Only load checkpoints you trust.


In [ ]:
source_name = "user" if USE_MY_DATA else "demo"
checkpoint_path = ARTIFACTS / f"08_imagefolder_{source_name}.pt"
checkpoint = {"state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
              "architecture": "TinyCNN", "in_channels": 3,
              "classes": class_names, "class_to_idx": class_to_idx,
              "image_size": IMAGE_SIZE, "input_range": [0.0, 1.0], "color_mode": "RGB"}
torch.save(checkpoint, checkpoint_path)

def predict_file(path):
    problem = inspect_image(Path(path))
    if problem is not None:
        raise ValueError(problem)
    with Image.open(path) as image:
        tensor = transform(image.convert("RGB")).unsqueeze(0).to(device)
    model.eval()
    with torch.inference_mode():
        probabilities = model(tensor).softmax(dim=1)[0].cpu()
    return pd.Series(probabilities.numpy(), index=class_names).sort_values(ascending=False)

example_path, expected_index = image_sets["test"].samples[0]
print("Expected:", class_names[expected_index], "| file:", example_path)
display(predict_file(example_path).head())
print("Saved:", checkpoint_path)


### Exercise: Inspect class imbalance

Compute each training class's fraction and the majority-class baseline. This baseline always predicts the most common training class.


In [ ]:
counts = np.bincount(image_sets["train"].targets, minlength=len(class_names))
print(pd.Series(counts / counts.sum(), index=class_names))
majority_class = int(counts.argmax())
print("Majority class:", class_names[majority_class])
# TODO: Measure this fixed predictor on the validation labels.


<details><summary>Sample answer — open after trying</summary>

```python
val_labels = np.asarray(image_sets['val'].targets)
print('Majority baseline validation accuracy:', (val_labels == majority_class).mean())
```
Choose the class from training data, then measure on validation. Do not select it using test labels.

</details>


### Exercise: Round-trip the checkpoint

Load the saved metadata into a new model. Check that predictions agree for the same image tensor.


In [ ]:
saved = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
restored = TinyCNN(num_classes=len(saved["classes"]), in_channels=saved["in_channels"])
restored.load_state_dict(saved["state_dict"])
restored.eval()
print("Restored classes:", saved["classes"])
# TODO: Compare logits from restored and model on xb[:2].


<details><summary>Sample answer — open after trying</summary>

```python
model.eval()
with torch.inference_mode():
    original_logits = model(xb[:2].to(device)).cpu()
    restored_logits = restored(xb[:2])
assert torch.allclose(original_logits, restored_logits, atol=1e-4, rtol=1e-4)
```
Small CPU/GPU floating-point differences are expected.

</details>


### Exercise: Plan a real dataset

Fill in this collection plan before switching USE_MY_DATA to True. Decide the independent group and a meaningful failure case.


In [ ]:
dataset_plan = {
    "classes": ["class_a", "class_b"],
    "independent_group": "one photographed physical object",
    "split_rule": "all photos of the same object stay together",
    "deployment_conditions": "describe lighting, camera, and background",
    "important_failure": "describe a costly or confusing mistake",
}
print(json.dumps(dataset_plan, indent=2))
# TODO: Replace these placeholders with your own project decisions.


<details><summary>Sample answer — open after trying</summary>

For leaf species, group by individual plant, collect multiple lighting conditions, and reserve different plants for test. Check that backgrounds or collection folders do not reveal labels. More copies of the same photograph do not add independent evidence.

</details>


**Checkpoint:** prepare train/val/test folders with consistent labels, detect corrupt files,
explain why hashes miss near-duplicates, and restore a saved model with its class order.

Next: [09 — Evaluation, imbalance, and calibration](09_evaluation_imbalance_calibration.ipynb).
References: [ImageFolder](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.ImageFolder.html),
[Pillow image validation](https://pillow.readthedocs.io/en/stable/reference/Image.html),
[saving and loading models](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html).
